# Лабораторная работа 3 CLIP & Zero-shot classification & Few-shot NER & Semantic deduplication

Результатом лабораторной работы является отчет. Мы предпочитаем принимать отчеты в формате ноутбуков IPython (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете также должен быть код, однако чем меньше кода, тем лучше всем: нам — меньше проверять, вам — проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.

Мы уверены, что выполнение лабораторных работ занимает значительное время, поэтому не рекомендуем оставлять их на последний вечер перед сдачей.



Штош...

Обучать модельки дорого: для каждой задачи надо собирать данные, учить модели, тратить время. Еще надо тратить время на дообучение моделей, например, ваш заказчик требовал от вас делать классификацию кошечек и собачек, а теперь ему нужна классификации на три класс, добавился ежик. Получается довольно много манипуляций и чтобы их не делать, множество задач можно решать в zero-shot или few-shot режимах, когда модели для распознавания нового класса достаточно нескольких примеров (обычно это единицы или десятки), либо вообще модель может работать с произвольным множеством классов. Один из ярких представителей такого семейства моделей: CLIP.

## Задание 1. CLIP (3 балла + бонусные)

Прочитайте [статью про CLIP](https://arxiv.org/abs/2103.00020) и письменно суммаризуйте информацию о модели, а именно:
- какая архитектура модели?
- как модель обучалась?
- какой размер обучающей выборки и каким образом авторы ее получили?
- как делать zero-shot классификацию на ImageNet, используя CLIP
- как дофайнтюнили авторы CLIP под конкретную задачу классификации?
- всегда ли CLIP-модели работают лучше на задаче классификации, если нет, то где есть обратный пример в статье?
- расскажите про робастость клипа к сдвигам в распределении (см. пример с бананами)?

Так же за дополнительные комментарии, интересные замечания по содержанию статьи могут быть бонусные баллы

In [ ]:
# YOUR THOUGHTS

## Задание 2. Zero-shot классификация на CIFAR10 (6 баллов)

Давайте попробуем применить CLIP с его 0-shot классификацией на задаче CIFAR10, а именно давайте попробуем сделать предсказания на валидационной выборке и выбить хорошие метрики.

In [ ]:
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from torchvision.datasets import CIFAR10
from transformers import AutoTokenizer, CLIPProcessor, CLIPModel

In [ ]:
# скачиваем датку
train_dataset = CIFAR10("./cifar10", train=True, download=True)
val_dataset = CIFAR10("./cifar10", train=False, download=True)

In [ ]:
fig, axs = plt.subplots(5, 5, figsize=(6, 6))

for i, ax in enumerate(axs.flatten()):
    image, label = train_dataset[i]
    ax.imshow(image)
    ax.set_title(f"{train_dataset.classes[label]}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model.eval();

In [ ]:
def cifar10_collate_fn(batch):
    images, labels = zip(*batch)
    images = [processor(images=image, return_tensors="pt")["pixel_values"] for image in images]
    images = torch.cat(images, 0)
    labels = torch.tensor(labels)
    return images, labels

batch_size = # YOUR CODE
train_loader = # YOUR CODE
val_loader = # YOUR CODE

In [ ]:
# пример доставание клиповых картиночных фич
batch = next(iter(val_loader))
with torch.no_grad():
    out = model.get_image_features(batch[0].to(device))
out.shape

Задание2.1 (1 балл): Извлеките картиночные фичи с обучающей и валидационной выборок. Для удобства, смотрите формат выходных данных в ассертах ниже.

In [ ]:
def extract_embddings(model, loader, device=device) -> tuple[torch.Tensor, torch.Tensor]:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
train_embeds, train_labels = extract_embddings(model, train_loader)
val_embeds, val_labels = extract_embddings(model, val_loader)

In [ ]:
assert train_embeds.shape == torch.Size([len(train_dataset), model.config.projection_dim])
assert val_embeds.shape == torch.Size([len(val_dataset), model.config.projection_dim])

assert train_labels.shape == torch.Size([len(train_dataset)])
assert val_labels.shape == torch.Size([len(val_dataset)])

print("cool!")

Задание 2.2 (1 балл): Извлеките текстовые фичи с обучающей и валидационной выборок.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_inputs = tokenizer(<YOUR CODE HERE>, return_tensors="pt").to(device)

with torch.no_grad():
    label_emb = model.get_text_features(<YOUR CODE HERE>)

assert label_emb.shape == torch.Size([len(train_dataset.classes), model.config.projection_dim])

Задание 2.3 (1 балл): Сделайте 0-shot предсказания с помощью картиночных и текстовых фич на валидационном датасете и посчитайте accuracy (должно быть не меньше 0.9).

In [ ]:
val_similarity = # <YOUR CODE HERE> -- матрица близостей картиночных и текстовых фич
val_pred_labels = # <YOUR CODE HERE> -- предсказания

In [ ]:
zeroshot_val_acc = # <YOUR CODE>
print(f"zero-shot val accuracy: {zeroshot_val_acc:.2f}")

Задание 2.4 (2 балла): Результат выше выглядит неплохо, а что если мы еще CLIP подфайнтюним? Обучите слой логистической регрессии на базе картиночных фич клипа на обучающем датасете и сделайте предсказания и подсчитайте метрики на валидационной. При этом веса CLIPа оставьте замороженными. (точность так же должна быть не меньше 0.9)

Задание 2.5 (1 балл): Обучите поверх картиночных фич клипа KNN, подберите гиперпараметры и оцените качество.

Но иногда заказчику не хватает одной классификации на картинках, ему, например, срочно требуется научиться распознавать именнованные сущности в тексте 🤠 ([NER](https://en.wikipedia.org/wiki/Named-entity_recognition)).

А у вас на серверах по-прежнему бегает дообучение РНН-ок к прошлой лабе, поэтому железа критически не хватает. Чтобы хоть как-то выпутаться из этой передряги, придется использовать все тот же zero-shot или даже few-shot.

Итак, задача:

# Задание 3. NER в few-shot (4 балла + 4 дополнительных)

Вам нужно решить задачу выделения именованных сущностей в тексте без использования обучающих данных... Что? Да, у вашего заказчика нет данных для обучения, но ему надо решить его задачу. Повторяю, трейн сета нет, его использовать нельзя)))

Для таких случаев сейчас есть следующие подходы:
1. Генерировать синтетические данные чтобы дообучиться на них и выдать предсказания на запрошенных данных
2. Использовать высокоранговые модели в few-shot без дообучения и выдать предсказания на запрошенных данных

**!НЕЛЬЗЯ!** Использовать дообученные берты на задаче NER, так как этот датасет туда практически наверное - уже протек

Если в первом случае сложности скорее в правильной генережке данных для обучения, то во втором - нахождения мощностей для инференса больших моделей (Qwen2.5-Omni, QWQ-32B, DeepSeek-V3 671B, ...).

На обязательные баллы вам нужно реализовать один из предложенных способов получения ответа. Чтобы получить дополнительные - следует добить второй. Вы также можете предложить свой способ дообучения, но его придется обосновать.

Мы будем использовать [датасет с кагла](https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus/data?select=ner_dataset.csv)

In [1]:
import pandas as pd
import dataclasses

@dataclasses.dataclass
class TRecord:
    sentence: list[str]
    tags: list[str]

    def render_tagged(self):
        return " ".join((f"{w}[{t}]" if t != "O" else w) for w, t in zip(self.sentence, self.tags))

df = pd.read_csv("../datasets/ner_dataset.csv", encoding="latin1")
df['Sentence #'] = df['Sentence #'].ffill()
dataset = [
    TRecord(**data)
    for data in df.groupby('Sentence #').apply(lambda x: dict(sentence=x["Word"].tolist(), tags=x["Tag"].tolist()), include_groups=False).values
]
dataset[0].render_tagged()

'Thousands of demonstrators have marched through London[B-geo] to protest the war in Iraq[B-geo] and demand the withdrawal of British[B-gpe] troops from that country .'

Теги бывают следующие:

In [2]:
df["Tag"].unique()

array(['O', 'B-geo', 'B-gpe', 'B-per', 'I-geo', 'B-org', 'I-org', 'B-tim',
       'B-art', 'I-art', 'I-per', 'I-gpe', 'I-tim', 'B-nat', 'B-eve',
       'I-eve', 'I-nat'], dtype=object)

Если какая-то сущность занимает больше одного слова, то первое слово помечается тегом `B-...`, а следующие слова тегами `I-...`



Ваша задача, побить аккураси (точность предсказания тега) в 0.9 без дообучения на этих данных. При этом евалиться должен весь датасет!

Для начала можете предсказывать точность предсказания тегов на позициях, где ground truth не равен тегу "O", но в итоге у вас должно получиться решение, которое предсказывает все теги правильно (с ошибками в 10%), а не только именнованные

In [4]:
def my_cool_algo_accuracy(dataset: list[TRecord]):
    acc = []
    for record in dataset:
        acc.append(sum(map(lambda x: x == "O", record.tags)) / len(record.tags))
    return sum(acc) / len(acc)

assert (my_cool_acc := my_cool_algo_accuracy(dataset)) >= 0.9, f"My acc: {my_cool_acc:.4f}"

AssertionError: My acc: 0.8499

Продемонстрируйте правильность своего алгоритма, выведя по 2-3 примера с наименьшим, средним и наивысшим accuracy. Что по ним можно сказать?

In [1]:
# TODO: code here

Мы верим что эта задача потребует от вас умения работы с внешними API, которые вы должны найти самостоятельно, либо же с дообучением бертов на синтетических данных, что также может потребовать немало времени, поэтому заранее закладывайте риски с инференсом в выполнение задания. Удачи!

P.S. Подсказка: если вы будете обучаться в few-shot, то вся задача сводится к подбору промпта, который вам обеспечит максимальный аккураси

# Задание 4. Уникализация в рантайме через кластеризацию. (3 бонусных)

На семинаре мы обсуждали как делать дедупликацию по отранжированным айтемам в выдачи вашей RecSys системы. Если коротко то алгоритм выглядит следующим образом:

```
Вход:
1. айтемы в порядке релевантности запросу
2. у каждого айтема есть смысловой эмбед который кодирует его тайтл/картинку/превью(в случае видео)/короткое содержание(в случае документа)
3. threshold - порог семантической близости айтемов

Алгоритм:
1. result = []
2. for item in items:
3.     if min(dist(item.embeddind, resulted_item.embedding)) > threshold:
4.         result.append(item)
5. items = result

Выход:
1. Айтемы в порядке релевантности, где для каждого смыслового кластера оставлен представитель с наивысшей релевантностью
```

### Описание задачи

Мы - начинающий стартап в области ECom. От CPO пришло видение, что в текущем виде результаты текстового поиска в наш сервис содержат подозрительно много товаров с одинаковыми названиями. Поэтому к нам пришла задача - убрать смысловые дубли из выдачи на поисковый запрос. Но так как стартап начинащий, у нас еще не сошлось логирование, а сводить задачу нужно уже сейчас. Из-за этого мы решили сделать MVP решение на синтетических данных.

Задача:
- Сделать через few-shot себе датасет:
    1. попросить придумать 100 поисковых запросов в ecom сервис
    2. для каждого поискового запроса попросить придлумать выдачу из 10-15 **уникальных** названий товаров
    3. выбрать половину товаров и попросить продублировать их, меняя немного названия, в 2-5 товаров
- Для каждого названия товара получить ембединги через bert модель
- Внутри сессии запустить алгоритм уникализации на основе расстояния ембедингов
- Придумать метрику, которая бы давала 1 если в сессии с поисковым запросом были убраны только дублирующие товары (с оставлением одного представителя) и 0, если пофильтрованы все уникальные товары
- Подобрать threshold под нашу задачу на основе синтетических данных

Примеры, готового диалога на небольшое количество айтемов, которым вы можете вдохновится: https://disk.yandex.ru/d/NllyhWCR1MGOPg

In [ ]:
# TODO: code here